## Gym and RLHF Experiments

### Gym

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["font.size"] = 46
plt.rcParams["axes.labelsize"] = 58
plt.rcParams["axes.titlesize"] = 66
plt.rcParams["legend.fontsize"] = 46
plt.rcParams["xtick.labelsize"] = 44
plt.rcParams["ytick.labelsize"] = 44

fig, axes = plt.subplots(1, 3, figsize=(60, 22), dpi=400)  # Higher DPI for quality

def exponential_moving_average(data, alpha=0.1):
    """Apply exponential moving average smoothing (alpha = new-sample weight)."""
    if len(data) == 0:
        return data
    ema = np.empty_like(data, dtype=float)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# Define colors
colors = {
    "DART": "#2E86AB",      # Blue
    "PPO": "#A23B72",       # Purple/Pink
    "PPO Double Critic": "#F18F01"  # Orange
}

dart_line_color = "#333333"  # Dark gray for the vertical line
dart_line_style = {"color": dart_line_color, "linestyle": "--", "linewidth": 4.5, "alpha": 0.9, "zorder": 5}

# ========================
# Plot 1: MountainCar
# ========================
df_mountain = pd.read_csv("benchmark_results/mountaincar_ppo_dart.csv")
N_SEEDS=32
steps = df_mountain["global_step"].values

dart_mean = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return"].values
dart_min = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return__MIN"].values
dart_max = df_mountain["MountainCar-v0__dart_ppo__1__1768426160 - charts/episodic_return__MAX"].values

ppo_mean = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return"].values
ppo_min = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return__MIN"].values
ppo_max = df_mountain["MountainCar-v0__ppo__1__1764901656 - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps = steps[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

# compute sem properly using number of seeds when available; otherwise fallback to half-range
half_range_dart = (dart_max - dart_min) / 2.0
half_range_ppo  = (ppo_max  - ppo_min)  / 2.0
if N_SEEDS and N_SEEDS > 1:
    dart_sem = half_range_dart / np.sqrt(N_SEEDS)
    ppo_sem  = half_range_ppo  / np.sqrt(N_SEEDS)
else:
    dart_sem = half_range_dart
    ppo_sem  = half_range_ppo

dart_ema = exponential_moving_average(dart_mean, alpha=0.1)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.1)
dart_sem_ema = exponential_moving_average(dart_sem, alpha=0.2)
ppo_sem_ema = exponential_moving_average(ppo_sem, alpha=0.2)

axes[0].plot(steps, dart_ema, label="DART", color=colors["DART"], linewidth=6)
axes[0].fill_between(steps, dart_ema - dart_sem_ema, dart_ema + dart_sem_ema, color=colors["DART"], alpha=0.60, linewidth=0)
axes[0].plot(steps, ppo_ema, label="PPO", color=colors["PPO"], linewidth=6)
axes[0].fill_between(steps, ppo_ema - ppo_sem_ema, ppo_ema + ppo_sem_ema, color=colors["PPO"], alpha=0.60, linewidth=0)
axes[0].axvline(x=1.75e6, **dart_line_style)
axes[0].set_xlabel("Step", fontweight="bold")
axes[0].set_ylabel("Episodic Return", fontweight="bold")
axes[0].set_title("MountainCar-v0", fontweight="bold", pad=20)
axes[0].legend(loc="upper left", frameon=True, fancybox=True, shadow=True)
axes[0].grid(True, alpha=0.3)

# ========================
# Plot 2: Sparse Pendulum
# ========================
df_pendulum = pd.read_csv("benchmark_results/sparse_pendulum_ppo_dart_largeppo.csv")

steps_pend = df_pendulum["Step"].values
dart_mean = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return"].values
dart_min = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return__MIN"].values
dart_max = df_pendulum["wandb_run_name: dart_pendulum_sparse - charts/episodic_return__MAX"].values

ppo_mean = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return"].values
ppo_min = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return__MIN"].values
ppo_max = df_pendulum["wandb_run_name: ppo_pendulum_sparse - charts/episodic_return__MAX"].values

ppo_large_mean = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return"].values
ppo_large_min = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return__MIN"].values
ppo_large_max = df_pendulum["wandb_run_name: ppo_pendulum_sparse_large_critic - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean) & ~np.isnan(ppo_large_mean)
steps_pend = steps_pend[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]
ppo_large_mean, ppo_large_min, ppo_large_max = ppo_large_mean[mask], ppo_large_min[mask], ppo_large_max[mask]

half_range_dart = (dart_max - dart_min) / 2.0
half_range_ppo  = (ppo_max  - ppo_min)  / 2.0
half_range_ppo_large = (ppo_large_max - ppo_large_min) / 2.0

if N_SEEDS and N_SEEDS > 1:
    dart_sem = half_range_dart / np.sqrt(N_SEEDS)
    ppo_sem  = half_range_ppo  / np.sqrt(N_SEEDS)
    ppo_large_sem = half_range_ppo_large / np.sqrt(N_SEEDS)
else:
    dart_sem = half_range_dart
    ppo_sem  = half_range_ppo
    ppo_large_sem = half_range_ppo_large

dart_ema = exponential_moving_average(dart_mean, alpha=0.1)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.1)
ppo_large_ema = exponential_moving_average(ppo_large_mean, alpha=0.1)
dart_sem_ema = exponential_moving_average(dart_sem, alpha=0.2)
ppo_sem_ema = exponential_moving_average(ppo_sem, alpha=0.2)
ppo_large_sem_ema = exponential_moving_average(ppo_large_sem, alpha=0.2)

axes[1].plot(steps_pend, dart_ema, label="DART", color=colors["DART"], linewidth=6)
axes[1].fill_between(steps_pend, dart_ema - dart_sem_ema, dart_ema + dart_sem_ema, color=colors["DART"], alpha=0.60, linewidth=0)
axes[1].plot(steps_pend, ppo_ema, label="PPO", color=colors["PPO"], linewidth=6)
axes[1].fill_between(steps_pend, ppo_ema - ppo_sem_ema, ppo_ema + ppo_sem_ema, color=colors["PPO"], alpha=0.60, linewidth=0)
axes[1].plot(steps_pend, ppo_large_ema, label="PPO Double Critic", color=colors["PPO Double Critic"], linewidth=6)
axes[1].fill_between(steps_pend, ppo_large_ema - ppo_large_sem_ema, ppo_large_ema + ppo_large_sem_ema, color=colors["PPO Double Critic"], alpha=0.60, linewidth=0)
axes[1].axvline(x=200, **dart_line_style)
axes[1].set_xlabel("Step", fontweight="bold")
axes[1].set_ylabel("Episodic Return", fontweight="bold")
axes[1].set_title("Sparse Pendulum", fontweight="bold", pad=20)
axes[1].legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
axes[1].grid(True, alpha=0.3)

# ========================
# Plot 3: Sparse Humanoid
# ========================
df_humanoid = pd.read_csv("benchmark_results/sparse_humanoid_ppo_dart_largeppo.csv")

steps_hum = df_humanoid["global_step"].values
dart_mean = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return"].values
dart_min = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return__MIN"].values
dart_max = df_humanoid["wandb_run_name: dart_humanoid_sparse - charts/episodic_return__MAX"].values

ppo_mean = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return"].values
ppo_min = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return__MIN"].values
ppo_max = df_humanoid["wandb_run_name: ppo_humanoid_sparse - charts/episodic_return__MAX"].values

mask = ~np.isnan(dart_mean) & ~np.isnan(ppo_mean)
steps_hum = steps_hum[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

half_range_dart = (dart_max - dart_min) / 2.0
half_range_ppo  = (ppo_max  - ppo_min)  / 2.0
if N_SEEDS and N_SEEDS > 1:
    dart_sem = half_range_dart / np.sqrt(N_SEEDS)
    ppo_sem  = half_range_ppo  / np.sqrt(N_SEEDS)
else:
    dart_sem = half_range_dart
    ppo_sem  = half_range_ppo

dart_ema = exponential_moving_average(dart_mean, alpha=0.01)
ppo_ema = exponential_moving_average(ppo_mean, alpha=0.01)
dart_sem_ema = exponential_moving_average(dart_sem, alpha=0.02)
ppo_sem_ema = exponential_moving_average(ppo_sem, alpha=0.02)

axes[2].plot(steps_hum, dart_ema, label="DART", color=colors["DART"], linewidth=6)
axes[2].fill_between(steps_hum, dart_ema - dart_sem_ema, dart_ema + dart_sem_ema, color=colors["DART"], alpha=0.45, linewidth=0)
axes[2].plot(steps_hum, ppo_ema, label="PPO", color=colors["PPO"], linewidth=6)
axes[2].fill_between(steps_hum, ppo_ema - ppo_sem_ema, ppo_ema + ppo_sem_ema, color=colors["PPO"], alpha=0.45, linewidth=0)
axes[2].axvline(x=2.8e7, **dart_line_style)
axes[2].set_xlabel("Step", fontweight="bold")
axes[2].set_ylabel("Episodic Return", fontweight="bold")
axes[2].set_title("Sparse Humanoid", fontweight="bold", pad=20)
axes[2].legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_results_latex/dart_vs_ppo_comparison.png", bbox_inches="tight")
print("Plots saved as 'dart_vs_ppo_comparison.png' and 'dart_vs_ppo_comparison.png'")
plt.show()

Plots saved as 'dart_vs_ppo_comparison.png' and 'dart_vs_ppo_comparison.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_26814/1081914259.py:189: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### RLHF

In [ ]:
### RLHF
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["font.size"] = 46
plt.rcParams["axes.labelsize"] = 58
plt.rcParams["axes.titlesize"] = 66
plt.rcParams["legend.fontsize"] = 46
plt.rcParams["xtick.labelsize"] = 44
plt.rcParams["ytick.labelsize"] = 44

# create figure
fig, ax = plt.subplots(1, 1, figsize=(22, 18), dpi=200)

def exponential_moving_average(data, alpha=0.05):
    """Apply exponential moving average smoothing."""
    ema = np.zeros_like(data)
    if len(data) == 0:
        return ema
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

# colours etc.
colors = {"DART": "#2E86AB", "PPO": "#A23B72"}
dart_line_color = "#333333"
dart_line_style = {"color": dart_line_color,
                   "linestyle": "--",
                   "linewidth": 3.5,
                   "alpha": 0.8,
                   "zorder": 5}

# ========================
# LLM PPO vs DART
# ========================
df = pd.read_csv("benchmark_results/LLM_ppo_dart_preliminary.csv")

# set to actual number of seeds if known (e.g. 3, 5). If None, use half-range (wider)
N_SEEDS = 10

steps = df["train/episode"].values
dart_mean = df["dart_enabled: true - train/objective/scores"].astype(float).values
dart_min  = df["dart_enabled: true - train/objective/scores__MIN"].astype(float).values
dart_max  = df["dart_enabled: true - train/objective/scores__MAX"].astype(float).values

ppo_mean = df["dart_enabled: false - train/objective/scores"].astype(float).values
ppo_min  = df["dart_enabled: false - train/objective/scores__MIN"].astype(float).values
ppo_max  = df["dart_enabled: false - train/objective/scores__MAX"].astype(float).values

mask = (~np.isnan(dart_mean) & ~np.isnan(ppo_mean) &
        ~np.isnan(dart_min) & ~np.isnan(dart_max) &
        ~np.isnan(ppo_min) & ~np.isnan(ppo_max))
steps = steps[mask]
dart_mean, dart_min, dart_max = dart_mean[mask], dart_min[mask], dart_max[mask]
ppo_mean, ppo_min, ppo_max = ppo_mean[mask], ppo_min[mask], ppo_max[mask]

half_range_dart = (dart_max - dart_min) / 2.0
half_range_ppo  = (ppo_max  - ppo_min)  / 2.0

if N_SEEDS and N_SEEDS > 1:
    dart_sem = half_range_dart / np.sqrt(N_SEEDS)
    ppo_sem  = half_range_ppo  / np.sqrt(N_SEEDS)
else:
    # fallback: use half-range as "SEM" (visual, not statistically exact)
    dart_sem = half_range_dart
    ppo_sem  = half_range_ppo

# smooth mean and sem
dart_ema     = exponential_moving_average(dart_mean, alpha=0.05)
ppo_ema      = exponential_moving_average(ppo_mean, alpha=0.05)
dart_sem_ema = exponential_moving_average(dart_sem, alpha=0.10)
ppo_sem_ema  = exponential_moving_average(ppo_sem, alpha=0.10)

# plot with stronger shading for visibility
ax.plot(steps, dart_ema, label="DART", color=colors["DART"], linewidth=5)
ax.fill_between(steps, dart_ema - dart_sem_ema, dart_ema + dart_sem_ema,
                color=colors["DART"], alpha=0.35, linewidth=0)

ax.plot(steps, ppo_ema, label="PPO", color=colors["PPO"], linewidth=5)
ax.fill_between(steps, ppo_ema - ppo_sem_ema, ppo_ema + ppo_sem_ema,
                color=colors["PPO"], alpha=0.35, linewidth=0)

ax.axvline(x=20000, **dart_line_style)

ax.set_xlabel("Episode", fontweight="bold")
ax.set_ylabel("Reward Score", fontweight="bold")
ax.legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_results_latex/dart_vs_ppo_llm.png", bbox_inches="tight")
print("Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.png'")
plt.show()

Plots saved as 'dart_vs_ppo_llm.png' and 'dart_vs_ppo_llm.pdf'


/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_26814/1166428641.py:100: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## dm_control plotting

### PPO vs. DART

In [3]:
# Cell: Fetch + cache histories (run once, then skip)

import os
import pandas as pd
import numpy as np
import wandb
from concurrent.futures import ThreadPoolExecutor, as_completed

WANDB_PROJECT = "aifgen/dm_control_ppo_vs_dart"
METRIC = "charts/episodic_return"
CACHE_FILE = "dm_control_histories.parquet"
N_SAMPLES = 2000  # subsample per run
N_WORKERS = 10    # parallel threads

if os.path.exists(CACHE_FILE):
    print(f"Loading cached data from {CACHE_FILE}")
    df = pd.read_parquet(CACHE_FILE)
else:
    print("Fetching run histories from wandb API...")
    api = wandb.Api()
    runs = api.runs(WANDB_PROJECT, per_page=300)
    run_list = list(runs)
    total = len(run_list)
    print(f"Found {total} runs")

    # Filter out non-finished runs upfront to avoid ValueError on .history()
    valid_runs = [r for r in run_list if r.state in ("finished", "crashed")]
    skipped = total - len(valid_runs)
    print(f"Skipping {skipped} runs (still running/failed) — using {len(valid_runs)} finished/crashed runs")

    def fetch_one(run):
        env_id = run.config.get("env_id", "")
        exp_name = run.config.get("exp_name", "")
        seed = run.config.get("seed", 0)
        if not env_id or not exp_name:
            return None
        try:
            hist = run.history(keys=[METRIC, "global_step"], samples=N_SAMPLES, pandas=True)
            if hist.empty:
                return None
            hist = hist.dropna(subset=[METRIC, "global_step"])
            if hist.empty:
                return None
            hist["env_id"] = env_id
            hist["exp_name"] = exp_name
            hist["seed"] = seed
            return hist
        except (ValueError, Exception) as e:
            print(f"\n  Warning: skipping {run.id} ({run.state}): {e}")
            return None

    all_data = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(fetch_one, run): i for i, run in enumerate(valid_runs)}
        for future in as_completed(futures):
            i = futures[future]
            result = future.result()
            if result is not None:
                all_data.append(result)
            print(f"\r[{len(all_data)} done / {i+1} processed / {len(valid_runs)} total]   ", end="", flush=True)

    print(f"\nFetched {len(all_data)} runs")
    df = pd.concat(all_data, ignore_index=True)

print(f"Dataset: {len(df)} rows | {df['env_id'].nunique()} envs | algos: {df['exp_name'].unique().tolist()}")

Loading cached data from dm_control_histories.parquet
Dataset: 2690000 rows | 45 envs | algos: ['ppo_double_dm_control', 'dart_dm_control', 'ppo_dm_control']


In [4]:
!uv add pyarrow fastparquet
df.to_parquet(CACHE_FILE)
print(f"Cached to {CACHE_FILE}")

Resolved 212 packages in 9ms
Audited 121 packages in 0.69ms
Cached to dm_control_histories.parquet


In [5]:
df=df[df['exp_name'] != 'ppo_dm_control']

In [6]:
df['exp_name'].unique()

array(['ppo_double_dm_control', 'dart_dm_control'], dtype=object)

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# --- Configuration ---
EMA_WEIGHT_MEAN = 0.90
EMA_WEIGHT_SEM = 0.80

METRIC = "charts/episodic_return"
ALGO_ORDER = ["dart_dm_control", "ppo_double_dm_control"]
ALGO_COLORS = {"dart_dm_control": "#2E86AB", "ppo_double_dm_control": "#F18F01"}
ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO"}

# How many plots per figure? 5 is standard for a full-width ICLR page
CHUNK_SIZE = 5  

def ema_smooth(values, weight=0.6):
    smoothed = np.zeros_like(values, dtype=float)
    if len(values) == 0:
        return smoothed
    smoothed[0] = values[0]
    for i in range(1, len(values)):
        smoothed[i] = weight * smoothed[i - 1] + (1 - weight) * values[i]
    return smoothed

def millions_formatter(x, pos):
    if x == 0:
        return "0"
    return f"{x * 1e-6:g}M"

sns.set_theme(style="whitegrid", rc={"grid.alpha": 0.5, "axes.edgecolor": "#cccccc"})

envs = sorted(df["env_id"].unique())
n_envs = len(envs)

# --- Chunking Logic ---
for chunk_idx in range(0, n_envs, CHUNK_SIZE):
    chunk_envs = envs[chunk_idx : chunk_idx + CHUNK_SIZE]
    
    # Create a 1-row figure. Figsize (14, 2.8) looks great stretched across \textwidth
    fig, axes = plt.subplots(1, CHUNK_SIZE, figsize=(14, 2.8), squeeze=False)
    
    for idx, env in enumerate(chunk_envs):
        ax = axes[0][idx]
        env_df = df[df["env_id"] == env]
        env_step_max = env_df["global_step"].max()

        for algo in ALGO_ORDER:
            algo_df = env_df[env_df["exp_name"] == algo]
            if algo_df.empty:
                continue

            step_min, step_max = algo_df["global_step"].min(), algo_df["global_step"].max()
            bins = np.linspace(step_min, step_max, 200)
            
            algo_df = algo_df.copy()
            algo_df["step_bin"] = pd.cut(algo_df["global_step"], bins=bins, labels=bins[:-1])
            algo_df["step_bin"] = algo_df["step_bin"].astype(float)

            grouped = algo_df.groupby("step_bin")[METRIC]
            mean = grouped.mean().dropna()
            std = grouped.std().fillna(0).loc[mean.index]
            count = grouped.count().loc[mean.index].replace(0, np.nan)  

            sem = std.values / np.sqrt(count.values)
            sem = np.nan_to_num(sem, nan=0.0)

            steps = mean.index.values
            mean_vals = ema_smooth(mean.values, EMA_WEIGHT_MEAN)
            sem_vals = ema_smooth(sem, EMA_WEIGHT_SEM)

            color = ALGO_COLORS[algo]
            label = ALGO_LABELS[algo]
            
            ax.plot(steps, mean_vals, color=color, label=label, linewidth=2)
            ax.fill_between(steps, mean_vals - sem_vals, mean_vals + sem_vals, alpha=0.15, color=color)

        # Plot the 40% Dotted Line
        dart_label = "DART Activated" if idx == 0 else None
        ax.axvline(x=env_step_max * 0.3, color="#7f8c8d", linestyle=":", linewidth=2, zorder=1, label=dart_label)

        # Formatting
        short_name = env.replace("dm_control/", "").replace("-v0", "")
        ax.set_title(short_name, fontsize=12, fontweight="bold", pad=8)
        
        # Only add y-label to the leftmost plot to save horizontal space
        if idx == 0:
            ax.set_ylabel("Return", fontsize=11, fontweight="medium")
        else:
            ax.set_ylabel("")
            
        ax.set_xlabel("Steps", fontsize=11, fontweight="medium")
        ax.xaxis.set_major_formatter(FuncFormatter(millions_formatter))
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Clean up empty subplots if the chunk isn't a full 5 environments
    for idx in range(len(chunk_envs), CHUNK_SIZE):
        fig.delaxes(axes[0][idx])

    # Add legend ONLY to the first chunk to save space, or above every chunk if you prefer. 
    # Here we put it above the figure.
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=11,
               frameon=False, bbox_to_anchor=(0.5, 1.15))

    plt.tight_layout()
    
    # Save each chunk with a distinct filename
    part_num = (chunk_idx // CHUNK_SIZE) + 1
    save_name = f"dm_control_benchmark_main_pt{part_num}"
    
    plt.savefig(f"benchmark_results_latex/{save_name}.png", bbox_inches="tight")
    print(f"Saved: benchmark_results_latex/{save_name}.png")
    
    # Close the figure so matplotlib doesn't eat up all your RAM
    plt.close(fig)

Saved: benchmark_results_latex/dm_control_benchmark_main_pt1.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt2.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt3.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt4.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt5.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt6.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt7.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt8.pdf
Saved: benchmark_results_latex/dm_control_benchmark_main_pt9.pdf


In [8]:
# Cell 3: Final performance table + ranking

import pandas as pd
import numpy as np

METRIC = "charts/episodic_return"
LAST_N = 20  # average over last N logged steps per run

ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO"}

# For each (env, algo, seed), take mean of last LAST_N steps
final_records = []
for (env, algo, seed), grp in df.groupby(["env_id", "exp_name", "seed"]):
    grp_sorted = grp.sort_values("global_step")
    tail = grp_sorted[METRIC].tail(LAST_N)
    final_records.append({
        "env_id": env,
        "exp_name": algo,
        "seed": seed,
        "final_return": tail.mean(),
    })

final_df = pd.DataFrame(final_records)

# Per (env, algo): mean ± std across seeds
summary = final_df.groupby(["env_id", "exp_name"])["final_return"].agg(["mean", "std", "count"]).reset_index()
summary["label"] = summary["exp_name"].map(ALGO_LABELS)
summary["display"] = summary.apply(lambda r: f"{r['mean']:.1f} ± {r['std']:.1f}", axis=1)

# Pivot to table
table = summary.pivot(index="env_id", columns="label", values="display")
table.index = table.index.str.replace("dm_control/", "").str.replace("-v0", "")
table = table[["DART", "PPO"]]  # fix column order
table.index.name = "Environment"

print("=" * 80)
print("Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)")
print("=" * 80)
print(table.to_string())
print()

# Bold the best per env
mean_pivot = summary.pivot(index="env_id", columns="label", values="mean")
mean_pivot.index = mean_pivot.index.str.replace("dm_control/", "").str.replace("-v0", "")
mean_pivot = mean_pivot[["DART", "PPO"]]

best_per_env = mean_pivot.idxmax(axis=1)
print("Best algorithm per environment:")
print(best_per_env.to_string())
print()

# Win counts
win_counts = best_per_env.value_counts()
print("Win counts:")
print(win_counts.to_string())
print()

# Overall ranking: mean of per-env means
overall = mean_pivot.mean(axis=0).sort_values(ascending=False)
print("Overall ranking (mean of per-env mean final return):")
for rank, (algo, val) in enumerate(overall.items(), 1):
    print(f"  #{rank} {algo}: {val:.1f}")

# Also save as LaTeX
# latex = table.to_latex(caption="Final episodic return (mean $\\pm$ std, last 20 steps, 10 seeds)")
# with open("dm_control_final_table.tex", "w") as f:
#     f.write(latex)
# print("\nSaved LaTeX table to dm_control_final_table.tex")

Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)
label                             DART            PPO
Environment                                          
acrobot-swingup             28.9 ± 8.0    24.9 ± 11.3
acrobot-swingup_sparse       2.1 ± 1.5      1.7 ± 1.5
ball_in_cup-catch         843.6 ± 38.8   855.8 ± 39.7
cartpole-balance          758.6 ± 31.1   746.2 ± 27.6
cartpole-balance_sparse   968.8 ± 21.5   956.7 ± 33.1
cartpole-swingup          602.8 ± 26.0   600.1 ± 25.3
cartpole-swingup_sparse  357.9 ± 203.2  230.9 ± 258.1
cartpole-three_poles       148.7 ± 4.9    149.1 ± 4.9
cartpole-two_poles         201.4 ± 6.1    209.6 ± 8.3
cheetah-run               309.0 ± 29.3    71.8 ± 18.4
dog-fetch                    8.8 ± 1.9      9.3 ± 1.8
dog-run                     14.4 ± 3.3     16.9 ± 5.0
dog-stand                  59.2 ± 12.0    83.6 ± 20.2
dog-trot                    23.6 ± 4.2     23.2 ± 4.9
dog-walk                    30.6 ± 5.7     33.7 ± 4.0
finger-

### MC Critic Ablation

In [9]:
# Cell: Fetch + cache histories (run once, then skip)

import os
import pandas as pd
import numpy as np
import wandb
from concurrent.futures import ThreadPoolExecutor, as_completed

WANDB_PROJECT = "dm_control_ppo_vs_dart_ablations"
METRIC = "charts/episodic_return"
CACHE_FILE = "dm_control_histories_mc_critic.parquet"
N_SAMPLES = 2000  # subsample per run
N_WORKERS = 10    # parallel threads

if os.path.exists(CACHE_FILE):
    print(f"Loading cached data from {CACHE_FILE}")
    df = pd.read_parquet(CACHE_FILE)
else:
    print("Fetching run histories from wandb API...")
    api = wandb.Api()
    runs = api.runs(WANDB_PROJECT, per_page=300)
    run_list = list(runs)
    total = len(run_list)
    print(f"Found {total} runs")

    # Filter out non-finished runs upfront to avoid ValueError on .history()
    valid_runs = [r for r in run_list if r.state in ("finished", "crashed")]
    skipped = total - len(valid_runs)
    print(f"Skipping {skipped} runs (still running/failed) — using {len(valid_runs)} finished/crashed runs")

    def fetch_one(run):
        env_id = run.config.get("env_id", "")
        exp_name = run.config.get("exp_name", "")
        seed = run.config.get("seed", 0)
        if not env_id or not exp_name:
            return None
        try:
            hist = run.history(keys=[METRIC, "global_step"], samples=N_SAMPLES, pandas=True)
            if hist.empty:
                return None
            hist = hist.dropna(subset=[METRIC, "global_step"])
            if hist.empty:
                return None
            hist["env_id"] = env_id
            hist["exp_name"] = exp_name
            hist["seed"] = seed
            return hist
        except (ValueError, Exception) as e:
            print(f"\n  Warning: skipping {run.id} ({run.state}): {e}")
            return None

    all_data = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(fetch_one, run): i for i, run in enumerate(valid_runs)}
        for future in as_completed(futures):
            i = futures[future]
            result = future.result()
            if result is not None:
                all_data.append(result)
            print(f"\r[{len(all_data)} done / {i+1} processed / {len(valid_runs)} total]   ", end="", flush=True)

    print(f"\nFetched {len(all_data)} runs")
    df = pd.concat(all_data, ignore_index=True)

print(f"Dataset: {len(df)} rows | {df['env_id'].nunique()} envs | algos: {df['exp_name'].unique().tolist()}")

Loading cached data from dm_control_histories_mc_critic.parquet
Dataset: 1906000 rows | 48 envs | algos: ['dart_dm_control', 'ppo_double_dm_control', 'ppo_mc_critic_dm_control']


In [10]:
df.to_parquet(CACHE_FILE)
print(f"Cached to {CACHE_FILE}")

Cached to dm_control_histories_mc_critic.parquet


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# --- Configuration ---
EMA_WEIGHT_MEAN = 0.90
EMA_WEIGHT_SEM = 0.80

METRIC = "charts/episodic_return"
ALGO_ORDER = ["dart_dm_control", "ppo_double_dm_control", "ppo_mc_critic_dm_control"]
ALGO_COLORS = {"dart_dm_control": "#2E86AB", "ppo_double_dm_control": "#F18F01", "ppo_mc_critic_dm_control": "#A23B72"}
ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO", "ppo_mc_critic_dm_control": "PPO w/ MC Critic"}

# How many plots per figure? 5 is standard for a full-width ICLR page
CHUNK_SIZE = 5  

def ema_smooth(values, weight=0.6):
    smoothed = np.zeros_like(values, dtype=float)
    if len(values) == 0:
        return smoothed
    smoothed[0] = values[0]
    for i in range(1, len(values)):
        smoothed[i] = weight * smoothed[i - 1] + (1 - weight) * values[i]
    return smoothed

def millions_formatter(x, pos):
    if x == 0:
        return "0"
    return f"{x * 1e-6:g}M"

sns.set_theme(style="whitegrid", rc={"grid.alpha": 0.5, "axes.edgecolor": "#cccccc"})

envs = sorted(df["env_id"].unique())
n_envs = len(envs)

# --- Chunking Logic ---
for chunk_idx in range(0, n_envs, CHUNK_SIZE):
    chunk_envs = envs[chunk_idx : chunk_idx + CHUNK_SIZE]
    
    # Create a 1-row figure. Figsize (14, 2.8) looks great stretched across \textwidth
    fig, axes = plt.subplots(1, CHUNK_SIZE, figsize=(14, 2.8), squeeze=False)
    
    for idx, env in enumerate(chunk_envs):
        ax = axes[0][idx]
        env_df = df[df["env_id"] == env]
        env_step_max = env_df["global_step"].max()

        for algo in ALGO_ORDER:
            algo_df = env_df[env_df["exp_name"] == algo]
            if algo_df.empty:
                continue

            step_min, step_max = algo_df["global_step"].min(), algo_df["global_step"].max()
            bins = np.linspace(step_min, step_max, 200)
            
            algo_df = algo_df.copy()
            algo_df["step_bin"] = pd.cut(algo_df["global_step"], bins=bins, labels=bins[:-1])
            algo_df["step_bin"] = algo_df["step_bin"].astype(float)

            grouped = algo_df.groupby("step_bin")[METRIC]
            mean = grouped.mean().dropna()
            std = grouped.std().fillna(0).loc[mean.index]
            count = grouped.count().loc[mean.index].replace(0, np.nan)  

            sem = std.values / np.sqrt(count.values)
            sem = np.nan_to_num(sem, nan=0.0)

            steps = mean.index.values
            mean_vals = ema_smooth(mean.values, EMA_WEIGHT_MEAN)
            sem_vals = ema_smooth(sem, EMA_WEIGHT_SEM)

            color = ALGO_COLORS[algo]
            label = ALGO_LABELS[algo]
            
            ax.plot(steps, mean_vals, color=color, label=label, linewidth=2)
            ax.fill_between(steps, mean_vals - sem_vals, mean_vals + sem_vals, alpha=0.15, color=color)

        # Plot the 40% Dotted Line
        dart_label = "DART Activated" if idx == 0 else None
        ax.axvline(x=env_step_max * 0.3, color="#7f8c8d", linestyle=":", linewidth=2, zorder=1, label=dart_label)

        # Formatting
        short_name = env.replace("dm_control/", "").replace("-v0", "")
        ax.set_title(short_name, fontsize=12, fontweight="bold", pad=8)
        
        # Only add y-label to the leftmost plot to save horizontal space
        if idx == 0:
            ax.set_ylabel("Return", fontsize=11, fontweight="medium")
        else:
            ax.set_ylabel("")
            
        ax.set_xlabel("Steps", fontsize=11, fontweight="medium")
        ax.xaxis.set_major_formatter(FuncFormatter(millions_formatter))
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Clean up empty subplots if the chunk isn't a full 5 environments
    for idx in range(len(chunk_envs), CHUNK_SIZE):
        fig.delaxes(axes[0][idx])

    # Add legend ONLY to the first chunk to save space, or above every chunk if you prefer. 
    # Here we put it above the figure.
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, fontsize=11,
               frameon=False, bbox_to_anchor=(0.5, 1.15))

    plt.tight_layout()
    
    # Save each chunk with a distinct filename
    part_num = (chunk_idx // CHUNK_SIZE) + 1
    save_name = f"dm_control_mc_benchmark_pt{part_num}"
    
    plt.savefig(f"benchmark_results_latex/{save_name}.png", bbox_inches="tight")
    print(f"Saved: {save_name}.png")
    
    # Close the figure so matplotlib doesn't eat up all your RAM
    plt.close(fig)

Saved: dm_control_mc_benchmark_pt1.pdf
Saved: dm_control_mc_benchmark_pt2.pdf
Saved: dm_control_mc_benchmark_pt3.pdf
Saved: dm_control_mc_benchmark_pt4.pdf
Saved: dm_control_mc_benchmark_pt5.pdf
Saved: dm_control_mc_benchmark_pt6.pdf
Saved: dm_control_mc_benchmark_pt7.pdf
Saved: dm_control_mc_benchmark_pt8.pdf
Saved: dm_control_mc_benchmark_pt9.pdf
Saved: dm_control_mc_benchmark_pt10.pdf


In [12]:
# Cell 3: Final performance table + ranking

import pandas as pd
import numpy as np

METRIC = "charts/episodic_return"
LAST_N = 20  # average over last N logged steps per run

ALGO_LABELS = {"dart_dm_control": "DART", "ppo_double_dm_control": "PPO", "ppo_mc_critic_dm_control": "PPO w/ MC Critic"}

# For each (env, algo, seed), take mean of last LAST_N steps
final_records = []
for (env, algo, seed), grp in df.groupby(["env_id", "exp_name", "seed"]):
    grp_sorted = grp.sort_values("global_step")
    tail = grp_sorted[METRIC].tail(LAST_N)
    final_records.append({
        "env_id": env,
        "exp_name": algo,
        "seed": seed,
        "final_return": tail.mean(),
    })

final_df = pd.DataFrame(final_records)

# Per (env, algo): mean ± std across seeds
summary = final_df.groupby(["env_id", "exp_name"])["final_return"].agg(["mean", "std", "count"]).reset_index()
summary["label"] = summary["exp_name"].map(ALGO_LABELS)
summary["display"] = summary.apply(lambda r: f"{r['mean']:.1f} ± {r['std']:.1f}", axis=1)

# Pivot to table
table = summary.pivot(index="env_id", columns="label", values="display")
table.index = table.index.str.replace("dm_control/", "").str.replace("-v0", "")
table = table[["DART", "PPO", "PPO w/ MC Critic"]]  # fix column order
table.index.name = "Environment"

print("=" * 80)
print("Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)")
print("=" * 80)
print(table.to_string())
print()

# Bold the best per env
mean_pivot = summary.pivot(index="env_id", columns="label", values="mean")
mean_pivot.index = mean_pivot.index.str.replace("dm_control/", "").str.replace("-v0", "")
mean_pivot = mean_pivot[["DART", "PPO", "PPO w/ MC Critic"]]

best_per_env = mean_pivot.idxmax(axis=1)
print("Best algorithm per environment:")
print(best_per_env.to_string())
print()

# Win counts
win_counts = best_per_env.value_counts()
print("Win counts:")
print(win_counts.to_string())
print()

# Overall ranking: mean of per-env means
overall = mean_pivot.mean(axis=0).sort_values(ascending=False)
print("Overall ranking (mean of per-env mean final return):")
for rank, (algo, val) in enumerate(overall.items(), 1):
    print(f"  #{rank} {algo}: {val:.1f}")

# Also save as LaTeX
# latex = table.to_latex(caption="Final episodic return (mean $\\pm$ std, last 20 steps, 10 seeds)")
# with open("dm_control_final_table.tex", "w") as f:
#     f.write(latex)
# print("\nSaved LaTeX table to dm_control_final_table.tex")

Final Episodic Return (mean ± std over seeds, averaged over last 20 steps)
label                             DART            PPO PPO w/ MC Critic
Environment                                                           
acrobot-swingup            26.4 ± 11.0    25.3 ± 11.9      20.3 ± 17.3
acrobot-swingup_sparse       1.9 ± 1.1      1.3 ± 1.0        1.4 ± 0.7
ball_in_cup-catch         833.4 ± 29.5   855.8 ± 39.7     858.2 ± 44.8
cartpole-balance          751.9 ± 16.8   743.5 ± 27.8     737.2 ± 27.3
cartpole-balance_sparse   960.3 ± 33.4   950.9 ± 34.8     953.0 ± 42.3
cartpole-swingup          595.8 ± 11.2   595.3 ± 26.4     590.1 ± 28.7
cartpole-swingup_sparse  294.1 ± 207.2  256.5 ± 259.9    417.6 ± 210.4
cartpole-three_poles       145.6 ± 4.4    149.9 ± 4.2     168.8 ± 14.8
cartpole-two_poles         192.6 ± 5.6    210.0 ± 9.3     210.3 ± 10.9
cheetah-run               302.5 ± 16.5    73.0 ± 20.5      73.9 ± 13.1
dog-fetch                    8.9 ± 2.0      8.8 ± 1.8       10.4 ± 2.7
do

### Value bias analysis

In [13]:
import os, time
import pandas as pd
import numpy as np
import wandb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from concurrent.futures import ThreadPoolExecutor, as_completed

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
WANDB_PROJECT = "aifgen/dm_control_ppo_vs_dart_ablations"
BIAS_METRIC   = "analysis/value_bias"
RET_METRIC    = "charts/episodic_return"
CACHE_BIAS    = "cache_bias.parquet"
CACHE_RETURN  = "cache_return.parquet"
EXCLUDE_EXP   = "ppo_mc_critic_dm_control"
N_SAMPLES     = 2000
N_WORKERS     = 10
EPS           = 1e-8

# Fixed-boundary bins by training PROGRESS (0→1), not by data quantiles
PROGRESS_BINS   = [0.0, 0.25, 0.50, 0.75, 1.001]
BIN_LABELS      = ["0-25%", "25-50%", "50-75%", "75-100%"]

KEEP_ENVS_RAW = [
    "acrobot swingup",          # ← replaces cartpole balance
    "acrobot swingup_sparse",
    "cartpole swingup_sparse",
    "cartpole swingup",         # ← add this
    "hopper hop",               # ← replaces hopper top
    "cheetah run",
    "finger turn_easy",
    "finger turn_hard",
    "pendulum swingup",
]


def classify(name):
    n = (name or "").lower()
    if "dart" in n: return "DART"
    if "ppo"  in n: return "PPO"
    return "OTHER"

def clean_env(e):
    return (e or "").replace("dm_control/","").replace("-v0","").replace("-"," ").replace("_"," ")

KEEP_ENVS = [clean_env(e) for e in KEEP_ENVS_RAW]

# ── FETCH ──────────────────────────────────────────────────────────────────────
def fetch_runs(metric, cache_file):
    if os.path.exists(cache_file):
        log(f"Cache hit: {cache_file}")
        return pd.read_parquet(cache_file)

    log(f"Fetching metric={metric} ...")
    api = wandb.Api(timeout=120)
    run_meta = get_run_list(api)

    # Filter here using metadata (no API call per run)
    valid_meta = [
        m for m in run_meta
        if m["state"] in ("finished", "crashed")
        and m["exp_name"] != EXCLUDE_EXP
        and m["env_id"]
        and m["exp_name"]
    ]
    log(f"Using {len(valid_meta)} runs")

    def fetch_one(meta):
        try:
            run = api.run(f"{WANDB_PROJECT}/{meta['id']}")
            hist = run.history(keys=[metric], x_axis="_step",
                               samples=N_SAMPLES, pandas=True)
            if hist is None or hist.empty or metric not in hist.columns:
                return None
            hist = hist.dropna(subset=[metric])
            if len(hist) < 5:
                return None

            hist = hist.sort_values("_step").reset_index(drop=True)
            n = len(hist)
            hist["progress"] = np.arange(n) / max(n - 1, 1)
            hist["bin"] = pd.cut(
                hist["progress"],
                bins=PROGRESS_BINS,
                labels=BIN_LABELS,
                include_lowest=True,
                right=False,
            )
            hist["env_clean"] = clean_env(meta["env_id"])
            hist["exp_name"]  = meta["exp_name"]
            hist["method"]    = classify(meta["exp_name"])
            hist["seed"]      = meta["seed"]

            return hist[["_step", "progress", "bin", metric,
                         "env_clean", "exp_name", "method", "seed"]]
        except Exception as e:
            print(f"\n  ERROR {meta['id']}: {e}", flush=True)
            return None

    all_data = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(fetch_one, m): i for i, m in enumerate(valid_meta)}
        for fut in as_completed(futures):
            out = fut.result()
            if out is not None:
                all_data.append(out)
            print(f"\r  collected {len(all_data)} / {len(valid_meta)}", end="", flush=True)
    print()

    if not all_data:
        raise RuntimeError(f"No valid runs for metric: {metric}")

    df = pd.concat(all_data, ignore_index=True)
    df.to_parquet(cache_file, index=False)
    log(f"Cached {len(df)} rows -> {cache_file}")
    return df


# ── LOAD ───────────────────────────────────────────────────────────────────────
df_bias = fetch_runs(BIAS_METRIC, CACHE_BIAS)
df_ret  = fetch_runs(RET_METRIC,  CACHE_RETURN)

for df in [df_bias, df_ret]:
    df.dropna(subset=["bin"], inplace=True)

df_bias = df_bias[df_bias["method"].isin(["DART","PPO"]) & df_bias["env_clean"].isin(KEEP_ENVS)].copy()
df_ret  = df_ret [df_ret ["method"].isin(["DART","PPO"]) & df_ret ["env_clean"].isin(KEEP_ENVS)].copy()

log(f"Bias: {len(df_bias)} rows | {df_bias['env_clean'].nunique()} envs")
log(f"Ret:  {len(df_ret)}  rows | {df_ret ['env_clean'].nunique()} envs")
log(f"Methods in bias: {df_bias['method'].unique()}")
log(f"Methods in ret:  {df_ret ['method'].unique()}")

missing = [e for e in KEEP_ENVS if e not in df_bias["env_clean"].unique()]
if missing:
    log(f"WARNING: these KEEP_ENVS not found in bias data: {missing}")
    log(f"Available: {sorted(df_bias['env_clean'].unique())}")

# ── SMOOTH RETURNS per run (sparse signal) ─────────────────────────────────────
def smooth_returns(grp):
    grp = grp.sort_values("progress").copy()
    n = len(grp)
    if n < 5:
        return grp
    win = max(5, n // 10)
    grp[RET_METRIC] = (grp[RET_METRIC]
                       .rolling(win, min_periods=1, center=True)
                       .mean()
                       .values)
    return grp

df_ret = df_ret.groupby(["env_clean","exp_name","seed"],
                         group_keys=False).apply(smooth_returns)

# ── AGGREGATE per (env, method, bin) ──────────────────────────────────────────
bias_agg = (df_bias
            .groupby(["env_clean","method","bin"], observed=True)[BIAS_METRIC]
            .mean().reset_index())

ret_agg = (df_ret
           .groupby(["env_clean","method","bin"], observed=True)[RET_METRIC]
           .mean().reset_index())

# ── DELTA TABLES ──────────────────────────────────────────────────────────────
def make_pivot(agg_df, metric, method_a, method_b, transform_fn, col_name):
    a = agg_df[agg_df["method"]==method_a].set_index(["env_clean","bin"])[metric]
    b = agg_df[agg_df["method"]==method_b].set_index(["env_clean","bin"])[metric]
    idx = a.index.intersection(b.index)
    tbl = pd.DataFrame({"a": a.loc[idx], "b": b.loc[idx]}).reset_index()
    tbl[col_name] = transform_fn(tbl["a"], tbl["b"])
    return tbl[["env_clean","bin",col_name]]

# (1) Bias magnitude improvement: + means DART closer to zero (uses abs ON PURPOSE)
bias_mag = make_pivot(
    bias_agg, BIAS_METRIC, "PPO", "DART",
    lambda ppo, dart: (ppo.abs() - dart.abs()) / (ppo.abs() + dart.abs() + EPS),
    "bias_mag"
)

# (2) Signed bias delta: + means PPO more optimistic (higher) than DART
bias_signed = make_pivot(
    bias_agg, BIAS_METRIC, "PPO", "DART",
    lambda ppo, dart: (ppo - dart) / (ppo.abs() + dart.abs() + EPS),
    "bias_signed"
)

# (3) Return improvement: + means DART higher return, symmetric norm (stable when PPO~0)
ret_delta = make_pivot(
    ret_agg, RET_METRIC, "DART", "PPO",
    lambda dart, ppo: (dart - ppo) / (dart.abs() + ppo.abs() + EPS),
    "ret_sym"
)

# ── HEATMAPS ──────────────────────────────────────────────────────────────────
# Sort envs by mean bias improvement (best DART envs first)
env_order = (bias_mag.groupby("env_clean")["bias_mag"].mean()
             .sort_values(ascending=False).index.tolist())
# Only keep envs present in bias data
env_order = [e for e in env_order if e in KEEP_ENVS]

def to_heat(df, val_col):
    h = df.pivot(index="env_clean", columns="bin", values=val_col)
    # reindex to consistent env order and bin order
    h = h.reindex(index=[e for e in env_order if e in h.index], columns=BIN_LABELS)
    return h

heat_bias_mag    = to_heat(bias_mag,    "bias_mag")
heat_bias_signed = to_heat(bias_signed, "bias_signed")
heat_ret         = to_heat(ret_delta,   "ret_sym")

log(f"Heatmap shape: {heat_bias_mag.shape}")
print("Bias mag heatmap:\n", heat_bias_mag.round(2))
print("Return heatmap:\n",   heat_ret.round(2))

# ── SCATTER: time-aligned (env, bin) points ───────────────────────────────────
scatter = (bias_mag
           .merge(ret_delta, on=["env_clean","bin"], how="inner")
           .dropna(subset=["bias_mag","ret_sym"]))
scatter = scatter[scatter["env_clean"].isin(KEEP_ENVS)].copy()
log(f"Scatter: {len(scatter)} points across {scatter['env_clean'].nunique()} envs")

# Per-phase correlation (across envs within each phase)
phase_corr = []
for b in BIN_LABELS:
    g = scatter[scatter["bin"] == b]
    if len(g) >= 4:
        r, p = stats.pearsonr(g["bias_mag"], g["ret_sym"])
        phase_corr.append((b, r, p, len(g)))
    else:
        phase_corr.append((b, np.nan, np.nan, len(g)))
# ── FIGURE: A + B only ────────────────────────────────────────────────────────
FONT = 14

sns.set_style("white")
plt.rcParams.update({
    "font.family":           "sans-serif",
    "font.size":              FONT,
    "axes.titlesize":         FONT + 2,
    "axes.labelsize":         FONT,
    "xtick.labelsize":        FONT - 1,
    "ytick.labelsize":        FONT - 1,
    "legend.fontsize":        FONT,
    "legend.title_fontsize":  FONT,
    "text.usetex":            False,
})

cmap = sns.diverging_palette(10, 133, s=80, l=45, as_cmap=True)
HM_KW = dict(
    cmap=cmap, center=0, vmin=-1.0, vmax=1.0,
    linewidths=1.0, linecolor="white",
    annot=True, fmt=".2f",
    annot_kws={"size": FONT - 1, "weight": "bold"},
)

n_envs = len(heat_bias_mag)
fig_ab, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(20, n_envs * 0.72 + 3.0),
    gridspec_kw={"wspace": 0.40},
)
fig_ab.patch.set_facecolor("white")

# Panel A
sns.heatmap(heat_bias_mag, ax=ax1, **HM_KW,
            cbar_kws={"label": "Sym. norm. bias reduction", "shrink": 0.75, "pad": 0.02})
ax1.set_title(
    r"(A)  $\frac{|\hat{V}_{PPO}| - |\hat{V}_{DART}|}{|\hat{V}_{PPO}| + |\hat{V}_{DART}|}$"
    "\n$+$ = DART closer to zero",
    fontweight="bold", pad=12,
)
ax1.set_xlabel("Training Phase", labelpad=8)
ax1.set_ylabel("Environment", labelpad=8)
for i, env in enumerate(heat_bias_mag.index):
    if heat_bias_mag.loc[env].dropna().gt(0).all():
        ax1.annotate("*", xy=(len(BIN_LABELS) + 0.12, i + 0.5),
                     xycoords="data", fontsize=FONT + 3, color="#1a7a1a",
                     va="center", annotation_clip=False, fontweight="bold")

# Panel B
sns.heatmap(heat_ret, ax=ax2, **HM_KW,
            cbar_kws={"label": "Sym. norm. return improvement", "shrink": 0.75, "pad": 0.02})
ax2.set_title(
    r"(B)  $\frac{R_{DART} - R_{PPO}}{|R_{DART}| + |R_{PPO}|}$"
    "\n$+$ = DART higher return",
    fontweight="bold", pad=12,
)
ax2.set_xlabel("Training Phase", labelpad=8)
ax2.set_ylabel("")

fig_ab.tight_layout()
fig_ab.savefig("benchmark_results_latex/dart_bias_return_heatmaps.png",
               dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig_ab)
log("Saved: dart_bias_return_heatmaps.png")

# ── APPENDIX: signed bias ──────────────────────────────────────────────────────
fig2, ax_app = plt.subplots(figsize=(10, max(5, n_envs * 0.65 + 1.5)))
fig2.patch.set_facecolor("white")
sns.heatmap(heat_bias_signed, ax=ax_app, **HM_KW,
            cbar_kws={
                "label": r"$(\hat{V}_{PPO} - \hat{V}_{DART})\,/\,(|\hat{V}_{PPO}| + |\hat{V}_{DART}|)$",
                "shrink": 0.8,
            })
ax_app.set_title(
    r"Signed bias direction: $\frac{\hat{V}_{PPO} - \hat{V}_{DART}}{|\hat{V}_{PPO}| + |\hat{V}_{DART}|}$"
    "\n$+$ = PPO more optimistic;   $-$ = DART more optimistic",
    fontweight="bold", pad=12,
)
ax_app.set_xlabel("Training Phase", labelpad=8)
ax_app.set_ylabel("Environment", labelpad=8)
plt.tight_layout()
fig2.savefig("benchmark_results_latex/appendix_signed_bias.png",
             dpi=220, bbox_inches="tight", facecolor="white")
plt.close(fig2)
log("Saved: appendix_signed_bias.png")

# ── PANEL C STANDALONE ────────────────────────────────────────────────────────
traj = (bias_mag
        .merge(ret_delta, on=["env_clean","bin"], how="inner")
        .dropna(subset=["bias_mag","ret_sym"]))
traj = traj[traj["env_clean"].isin(KEEP_ENVS)].copy()
traj["bin"] = pd.Categorical(traj["bin"], categories=BIN_LABELS, ordered=True)
traj = traj.sort_values(["env_clean","bin"])
envs_sorted = (bias_mag.groupby("env_clean")["bias_mag"].mean()
               .sort_values(ascending=False).index.tolist())
envs_sorted = [e for e in envs_sorted if e in traj["env_clean"].unique()]

SM_W     = 4.0
SM_H     = 4.0
N_COLS   = 7
n_sm_rows = int(np.ceil(len(envs_sorted) / N_COLS))

phase_palette = {
    "0-25%":   "#aecfe8",
    "25-50%":  "#5ba4cf",
    "50-75%":  "#1f6faa",
    "75-100%": "#0a3d6b",
}
phase_colors = [phase_palette[b] for b in BIN_LABELS]

SM_FONT      = 15
SM_TICK_FONT = 13
SM_AXIS_FONT = 14

fig_c = plt.figure(figsize=(N_COLS * SM_W + 1.0, n_sm_rows * SM_H + 2.5))
fig_c.patch.set_facecolor("white")

total_plots = len(envs_sorted)
last_row_plots = total_plots % N_COLS if total_plots % N_COLS != 0 else N_COLS
for idx, env in enumerate(envs_sorted):
    row_i = idx // N_COLS
    col_i = idx % N_COLS
    # Center last row if not full
    if row_i == n_sm_rows - 1 and last_row_plots != N_COLS:
        offset = (N_COLS - last_row_plots) // 2
        col_i += offset
    ax = fig_c.add_subplot(n_sm_rows, N_COLS, row_i * N_COLS + col_i + 1)

    grp   = traj[traj["env_clean"] == env].set_index("bin").reindex(BIN_LABELS)
    xs    = grp["bias_mag"].values.astype(float)
    ys    = grp["ret_sym"].values.astype(float)
    valid = ~(np.isnan(xs) | np.isnan(ys))
    vidx  = np.where(valid)[0]

    ax.fill_between([-1.05, 0], [-1.05]*2, [1.05]*2, color="#fde8e8", alpha=0.25, zorder=0)
    ax.fill_between([0, 1.05],  [-1.05]*2, [1.05]*2, color="#e8f5e8", alpha=0.25, zorder=0)

    if valid.sum() >= 2:
        ax.plot(xs[valid], ys[valid], color="#999", lw=1.8, zorder=1)

    if len(vidx) >= 2:
        x0, y0 = xs[vidx[-2]], ys[vidx[-2]]
        x1, y1 = xs[vidx[-1]], ys[vidx[-1]]
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="-|>", color="#444",
                                   lw=1.5, mutation_scale=16),
                    zorder=4)

    for j in range(len(BIN_LABELS)):
        if valid[j]:
            ax.scatter(xs[j], ys[j], s=120, color=phase_colors[j],
                       edgecolors="white", linewidths=0.8, zorder=3)

    ax.axhline(0, color="#bbb", lw=1.0, zorder=0)
    ax.axvline(0, color="#bbb", lw=1.0, zorder=0)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    ax.tick_params(labelsize=SM_TICK_FONT)
    ax.set_facecolor("#fafafa")
    for spine in ax.spines.values():
        spine.set_edgecolor("#ccc")

    ax.set_title(env, fontsize=SM_FONT, fontweight="bold", pad=6)

    if col_i == 0:
        ax.set_ylabel(r"$\Delta R$", fontsize=SM_AXIS_FONT, labelpad=5)
    else:
        ax.set_yticklabels([])

    if row_i == n_sm_rows - 1 or idx >= len(envs_sorted) - N_COLS:
        ax.set_xlabel(r"$\Delta|\mathrm{bias}|$", fontsize=SM_AXIS_FONT, labelpad=5)
    else:
        ax.set_xticklabels([])

for idx in range(len(envs_sorted), n_sm_rows * N_COLS):
    fig_c.add_subplot(n_sm_rows, N_COLS, idx + 1).set_visible(False)

fig_c.suptitle(
    r"(C)  Per-environment trajectory: $|\mathrm{bias}|$ reduction vs. return improvement"
    "\n(each point = one training phase;  arrow = time direction;  green = DART better on both axes)",
    fontsize=17, fontweight="bold", y=1.01,
)
legend_handles = [mpatches.Patch(color=phase_palette[b], label=b) for b in BIN_LABELS]
fig_c.legend(
    handles=legend_handles,
    title="Training phase",
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    fontsize=14, title_fontsize=14,
    frameon=True, facecolor="white", edgecolor="#bbb",
    handlelength=1.8,
)
fig_c.tight_layout()
fig_c.savefig("benchmark_results_latex/dart_panel_c_standalone.png",
              dpi=200, bbox_inches="tight", facecolor="white")
plt.close(fig_c)
log("Saved: dart_panel_c_standalone.png")


[10:48:54] Cache hit: cache_bias.parquet
[10:48:54] Cache hit: cache_return.parquet
[10:48:55] Bias: 348979 rows | 9 envs
[10:48:55] Ret:  350000  rows | 9 envs
[10:48:55] Methods in bias: ['DART' 'PPO']
[10:48:55] Methods in ret:  ['PPO' 'DART']
[10:48:55] Heatmap shape: (9, 4)
Bias mag heatmap:
 bin                      0-25%  25-50%  50-75%  75-100%
env_clean                                              
pendulum swingup          0.12   -0.01    0.55     0.96
hopper hop               -0.23    0.90   -0.09     0.86
cartpole swingup          0.10    0.13    0.47     0.54
acrobot swingup          -0.03    0.00    0.49     0.48
finger turn hard          0.03   -0.10    0.27     0.35
acrobot swingup sparse   -0.08    0.01    0.24     0.26
finger turn easy          0.12   -0.09    0.11     0.05
cartpole swingup sparse  -0.78   -0.01    0.71    -0.14
cheetah run              -0.03   -0.43   -0.44    -0.62
Return heatmap:
 bin                      0-25%  25-50%  50-75%  75-100%
env_clean   

/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_26814/1700579720.py:158: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ret = df_ret.groupby(["env_clean","exp_name","seed"],
/var/folders/vm/6_sk059d25x7g6b8w6f1_vc40000gn/T/ipykernel_26814/1700579720.py:296: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig_ab.tight_layout()


[10:48:56] Saved: dart_bias_return_heatmaps.png
[10:48:56] Saved: appendix_signed_bias.png
[10:48:58] Saved: dart_panel_c_standalone.png
